<a href="https://colab.research.google.com/github/AufanT/AufanT-BigData26_B_2411532011_AufanTaufiqurrahman/blob/main/Praktikum_4/BD_P04_2411532011_AufanTaufiqurrahman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BD_P04 — Praktikum 4 Big Data: NoSQL dan Pemodelan Data

| | |
|---|---|
| Nama | Aufan Taufiqurrahman |
| NIM | 2411532011 |
| Program Studi | S1 Informatika, Universitas Andalas |
| Dosen | Afdhal Dinilhak |

**Isi notebook:** K-1 s.d. K-9, Latihan 1–6, Tugas R.2 (data lain, cara lain), dan R.3 (Delta Lake dua hari).
R.1 dipenuhi oleh `benchmark_nosql.csv` dan tabel rekap di K-9.

**Cara pakai**
1. Runtime → Change runtime type → CPU. Jalankan sel dari atas secara berurutan (jangan melompat).
2. Pada tiap bagian K-2 sampai K-8 ada sel Markdown **"Dugaan saya"**. Isi dulu sebelum menjalankan bagian itu, jangan sesudah melihat hasil.
3. Untuk pengumpulan: Runtime → **Restart session and run all** pada runtime bersih, lalu unduh `.ipynb` dengan output.

**Catatan penyimpangan dari modul (disengaja)**
- Tabel Delta ditulis ke `/content/trips_delta` (disk lokal Colab), bukan langsung ke Drive. Penulisan Delta memakai operasi berkas yang sering gagal atau lambat di Drive. ZIP-nya disalin ke Drive setelah K-8.
- `trips_bersih/` dipulihkan dari `trips_bersih.zip` Praktikum 3 bila folder aslinya tidak ada di Drive.
- `trips_bersih/` dibaca lewat `pyarrow.dataset`, bukan `pd.read_parquet(folder)`. Data Praktikum 3 punya 340 baris dengan wilayah kosong (partisi `__HIVE_DEFAULT_PARTITION__`), dan cara kedua bisa gagal dengan `Cannot yet unify dictionaries with nulls`.

## K-1. Menyiapkan Lingkungan dan Data Contoh
Sel pertama khusus Colab (pasang pustaka + mount Drive).

In [1]:
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"
from google.colab import drive
drive.mount("/content/drive")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 24.1 MB/s eta 0:00:00
Mounted at /content/drive


### K-1b. Jalur cadangan: data tiruan bila `trips_bersih/` hilang
**LEWATI sel ini bila `trips_bersih/` Praktikum 3 masih ada** (di Drive atau sebagai `trips_bersih.zip`).
Bila dipakai, hasil percobaan harus dilaporkan sebagai data tiruan.

In [2]:
import numpy as np, pandas as pd, os, shutil

def bangkitkan_trips_sintetis(keluar, n=1_000_000, seed=42):
    rng = np.random.default_rng(seed)
    wilayah = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "EWR"]
    zona_ids = np.arange(1, 264)
    wilayah_zona = rng.choice(wilayah, size=len(zona_ids), p=[0.35, 0.25, 0.22, 0.10, 0.05, 0.03])
    zona = pd.DataFrame({"LocationID": zona_ids, "borough_naik": wilayah_zona})
    pop = np.clip(rng.zipf(1.6, size=len(zona_ids)).astype(float), 1, 60)
    pop /= pop.sum()
    pu = rng.choice(zona_ids, size=n, p=pop)
    do = rng.choice(zona_ids, size=n, p=pop)
    awal = pd.Timestamp("2023-01-01")
    pickup = awal + pd.to_timedelta(rng.integers(0, 31 * 24 * 60, n), unit="m")
    durasi = rng.integers(2, 90, n)
    dist = np.round(rng.gamma(2.0, 1.6, n).clip(0.05, 60), 2)
    fare = np.round(2.5 + dist * 2.8 + rng.gamma(2.0, 1.0, n), 2)
    tip = np.round(rng.gamma(1.5, 1.2, n), 2)
    pay = rng.choice([1, 2, 3, 4], size=n, p=[0.7, 0.27, 0.02, 0.01])
    d = pd.DataFrame({
        "tpep_pickup_datetime": pickup,
        "tpep_dropoff_datetime": pickup + pd.to_timedelta(durasi, unit="m"),
        "passenger_count": rng.integers(1, 5, n).astype("float64"),
        "trip_distance": dist, "PULocationID": pu, "DOLocationID": do,
        "payment_type": pay, "fare_amount": fare, "tip_amount": tip,
        "total_amount": np.round(fare + tip + 3.0, 2),
        "durasi_menit": durasi.astype("float64"),
        "nama_pembayaran": pd.Series(pay).map({1: "Kartu kredit", 2: "Tunai", 3: "Gratis", 4: "Sengketa"}).values,
    })
    d = d.merge(zona.rename(columns={"LocationID": "PULocationID"}), on="PULocationID", how="left")
    d["suhu_c"] = np.round(rng.normal(3, 5, n), 1)
    d["hujan_mm"] = np.round(rng.gamma(0.4, 0.8, n), 2)
    if os.path.exists(keluar):
        shutil.rmtree(keluar)
    d.to_parquet(keluar, partition_cols=["borough_naik"], index=False)
    return keluar

# Hanya bila trips_bersih/ hilang: buka komentar baris di bawah, lalu lanjut ke sel K-1.
# TRIPS_BERSIH_SINTETIS = bangkitkan_trips_sintetis("/content/trips_bersih_sintetis")

### K-1. Pustaka, jalur folder, pengukur waktu, dan data contoh

In [3]:
import os, time, json, sqlite3, random, shutil, glob, zipfile
import numpy as np, pandas as pd

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
TRIPS_BERSIH = globals().get("TRIPS_BERSIH_SINTETIS", f"{DIR_P3}/trips_bersih")  # hasil Praktikum 3 (K-10)
os.makedirs(DIR_SIMPAN, exist_ok=True)

# Di Praktikum 3 folder trips_bersih/ ditulis ke /content; yang masuk Drive hanya trips_bersih.zip.
# Bila foldernya tidak ada, pulihkan dari ZIP.
if not os.path.exists(TRIPS_BERSIH):
    ZIP_P3 = f"{DIR_P3}/trips_bersih.zip"
    if os.path.exists(ZIP_P3):
        TRIPS_BERSIH = "/content/trips_bersih"
        shutil.rmtree(TRIPS_BERSIH, ignore_errors=True)
        with zipfile.ZipFile(ZIP_P3) as z:
            z.extractall(TRIPS_BERSIH)
        print("trips_bersih/ dipulihkan dari ZIP:", ZIP_P3)

assert os.path.exists(TRIPS_BERSIH), "Folder trips_bersih/ tidak ada. Periksa DIR_P3 atau pakai K-1b."
if "TRIPS_BERSIH_SINTETIS" in globals():
    print("PERINGATAN: memakai DATA TIRUAN (K-1b). Laporkan sebagai data tiruan.")

# Pengukur waktu (dari Praktikum 1/K-3, versi ringkas)
catatan = []
waktu_mentah = {}   # tambahan: waktu tanpa pembulatan, dipakai untuk menghitung rasio

def ukur(label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 4)})
    waktu_mentah[label] = detik
    print(f"[{label}] {detik:.4f} s")
    return hasil

def ukur_median(label, fungsi, ulang=5):
    # tambahan: ulangi beberapa kali dan ambil median, untuk pengukuran yang sangat singkat
    daftar = []
    for _ in range(ulang):
        t0 = time.perf_counter()
        hasil = fungsi()
        daftar.append(time.perf_counter() - t0)
    detik = float(np.median(daftar))
    catatan.append({"langkah": label, "detik": round(detik, 6)})
    waktu_mentah[label] = detik
    print(f"[{label}] median {ulang}x: {detik:.6f} s")
    return hasil

import pyarrow.dataset as ds

def baca_trips():
    # Dibaca lewat pyarrow.dataset tanpa kamus (dictionary): pd.read_parquet(folder) gagal dengan
    # "Cannot yet unify dictionaries with nulls" bila ada partisi __HIVE_DEFAULT_PARTITION__ (wilayah kosong).
    return ds.dataset(TRIPS_BERSIH, format="parquet", partitioning="hive").to_table().to_pandas()

df = ukur("baca trips_bersih", baca_trips)

# Ganti nama kolom asli data TLC menjadi nama yang lebih mudah dibaca
NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}
df = df.rename(columns=NAMA_BARU)
n_wilayah_kosong = int(df["wilayah_jemput"].isna().sum())
print("baris dengan wilayah_jemput kosong:", n_wilayah_kosong)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)
print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])

# Contoh 300.000 baris untuk K-2 dan K-3 (agar tidak terlalu lama)
sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)

trips_bersih/ dipulihkan dari ZIP: /content/drive/MyDrive/BigData/Praktikum3/trips_bersih.zip
[baca trips_bersih] 4.8379 s
baris dengan wilayah_jemput kosong: 340
baris: 2,986,910 | kolom: 18


## K-2. Tabel Biasa vs Key-value

**Dugaan saya (isi SEBELUM menjalankan sel di bawah):**
- Cari satu id: cara yang menang = ............ , kira-kira ......... kali lipat dibanding yang paling lambat.
- Hitung rata-rata per wilayah: cara yang menang = ............ , kira-kira ......... kali lipat.

In [4]:
KOLOM = ["id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan",
         "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data = sample[KOLOM].copy()

# Model 1 - relasional: tabel SQLite
if os.path.exists("uji.db"):
    os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

# Model 2 - key-value: kamus (dict) dengan kunci "perjalanan:<id>", nilainya teks JSON
def bangun_kv():
    return {f"perjalanan:{r['id_perjalanan']}": json.dumps(r) for r in data.to_dict("records")}

kv = ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))

# Pertanyaan A: ambil SATU perjalanan berdasarkan id (200 id acak)
random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)

def cari_sql():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji]

def cari_kv():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji]

r1 = ukur("cari 200 perjalanan - SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan(id_perjalanan)")  # buat indeks
r2 = ukur("cari 200 perjalanan - SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan - key-value", cari_kv)
print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

# Pertanyaan B: hitung rata-rata total bayar per wilayah (melibatkan SEMUA data)
def hitung_sql():
    return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
                          FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall()

def hitung_kv():
    total, jumlah = {}, {}
    for teks in kv.values():
        d = json.loads(teks)          # setiap nilai harus dibuka dulu
        w = d["wilayah_jemput"]
        total[w] = total.get(w, 0) + d["total_bayar"]
        jumlah[w] = jumlah.get(w, 0) + 1
    return sorted([(w, jumlah[w], round(total[w] / jumlah[w], 2)) for w in jumlah],
                  key=lambda x: (-x[1], x[0]))

h1 = ukur("hitung per wilayah - SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah - key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)

[relasional: muat tabel] 4.0217 s
[key-value: bangun dict] 7.1633 s
jumlah key: 300000
[cari 200 perjalanan - SQL tanpa indeks] 4.2453 s
[cari 200 perjalanan - SQL dengan indeks] 0.0038 s
[cari 200 perjalanan - key-value] 0.0012 s
hasil sama? True
[hitung per wilayah - SQL GROUP BY] 0.1915 s
[hitung per wilayah - key-value (buka semua nilai)] 1.4724 s
[('Manhattan', 267034, 22.7), ('Queens', 27288, 70.34), ('Unknown', 3787, 39.13)]
hasil sama? True


## K-3. Document: Bentuk Data yang Fleksibel

**Dugaan saya (isi SEBELUM menjalankan):**
- Berapa dokumen (dari 20.000) yang TIDAK punya field `tip`? ......... (perkiraan persen: ...... %)
- Apakah database menolak dokumen ke-20.001 yang membawa field `cuaca`? ............

In [5]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
    dok = {
        "id_perjalanan": int(r.id_perjalanan),
        "waktu_jemput": r.waktu_jemput,
        "jemput": {"zona": int(r.zona_jemput), "wilayah": r.wilayah_jemput},
        "tujuan": {"zona": int(r.zona_tujuan)},
        "biaya": {"jarak_mil": float(r.jarak_mil),
                  "total": float(r.total_bayar),
                  "metode_bayar": r.metode_bayar},
    }
    # Tip hanya dicatat untuk pembayaran kartu -> field ini ada di sebagian dokumen saja
    if r.metode_bayar == "Kartu kredit":
        dok["biaya"]["tip"] = float(r.tip)
    return dok

db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]
tabel.insert_multiple(docs)
print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2, default=str))

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-16 10:40:39",
  "jemput": {
    "zona": 48,
    "wilayah": "Manhattan"
  },
  "tujuan": {
    "zona": 164
  },
  "biaya": {
    "jarak_mil": 1.55,
    "total": 19.32,
    "metode_bayar": "Kartu kredit",
    "tip": 3.22
  }
}


In [6]:
Q = Query()
n_manhattan_50 = len(tabel.search((Q.jemput.wilayah == "Manhattan") & (Q.biaya.total > 50)))
n_dok_tip = len(tabel.search(Q.biaya.tip.exists()))
n_dok_tanpa_tip = len(tabel.search(~Q.biaya.tip.exists()))
print("jemput di Manhattan & total > 50 :", n_manhattan_50)
print("dokumen yang punya field tip     :", n_dok_tip)
print("dokumen yang TIDAK punya tip     :", n_dok_tanpa_tip)

# Menambah field baru ("cuaca") pada satu dokumen - tanpa mengubah struktur apa pun
tabel.insert({"id_perjalanan": 999_999, "waktu_jemput": "2023-01-15 08:00:00",
              "jemput": {"zona": 161, "wilayah": "Manhattan"}, "tujuan": {"zona": 236},
              "biaya": {"jarak_mil": 1.2, "total": 14.5, "metode_bayar": "Kartu kredit", "tip": 3.0},
              "cuaca": {"hujan": True, "suhu_c": -2.0}})
n_dok_cuaca = len(tabel.search(Q.cuaca.exists()))
n_dok_total = len(tabel)
print("dokumen yang punya field cuaca   :", n_dok_cuaca)
print("total dokumen sekarang           :", n_dok_total)

jemput di Manhattan & total > 50 : 693
dokumen yang punya field tip     : 15906
dokumen yang TIDAK punya tip     : 4094
dokumen yang punya field cuaca   : 1
total dokumen sekarang           : 20001


## K-4. Penyimpanan per Kolom: DuckDB pada Parquet

**Dugaan saya (isi SEBELUM menjalankan):**
- DuckDB lebih cepat dari SQLite sekitar ......... kali lipat untuk hitung rata-rata per wilayah.
- Query yang difilter pada satu wilayah (partisi) lebih cepat/lambat dari hitungan penuh: ............

In [7]:
import duckdb

GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
con_d = duckdb.connect()

# VIEW = "jendela" bernama yang membaca Parquet langsung (tidak menyalin data).
# Di sinilah nama kolom asli TLC diberi nama yang mudah dibaca.
con_d.sql(f"""
    CREATE VIEW perjalanan AS
    SELECT tpep_pickup_datetime AS waktu_jemput,
           PULocationID AS zona_jemput,
           DOLocationID AS zona_tujuan,
           borough_naik AS wilayah_jemput,
           trip_distance AS jarak_mil,
           total_amount AS total_bayar,
           nama_pembayaran AS metode_bayar
    FROM read_parquet('{GLOB}', hive_partitioning=true)""")

SQL = """SELECT wilayah_jemput, COUNT(*) AS jumlah, ROUND(AVG(total_bayar), 2) AS rata_bayar
         FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC, wilayah_jemput"""
hasil_d = ukur("DuckDB - hitung per wilayah (langsung dari Parquet)", lambda: con_d.sql(SQL).df())
print(hasil_d)

# Pembanding: SQLite (menyimpan per baris) pada data yang sama
KOLOM_S = ["wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"]
if os.path.exists("baris.db"):
    os.remove("baris.db")
con_s = sqlite3.connect("baris.db")
ukur("SQLite - muat seluruh baris ke tabel", lambda: df[KOLOM_S].to_sql("perjalanan", con_s, index=False))
hasil_s = ukur("SQLite - hitung per wilayah", lambda: con_s.execute(
    """SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
       FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall())
print("hasil sama?", [tuple(r) for r in hasil_d.itertuples(index=False)] == [tuple(r) for r in hasil_s])

[DuckDB - hitung per wilayah (langsung dari Parquet)] 0.1158 s
               wilayah_jemput   jumlah  rata_bayar
0                   Manhattan  2659609       22.72
1                      Queens   270315       70.46
2                     Unknown    37609       39.24
3                    Brooklyn    15724       32.72
4                       Bronx     3074       34.44
5  __HIVE_DEFAULT_PARTITION__      340       74.72
6               Staten Island      211       74.07
7                         EWR       28      106.41
[SQLite - muat seluruh baris ke tabel] 9.2370 s
[SQLite - hitung per wilayah] 1.7315 s
hasil sama? False


**Pemeriksaan tambahan.** Data Praktikum 3 punya baris dengan wilayah kosong (ID zona 265). DuckDB membacanya sebagai `NULL`
(atau teks `__HIVE_DEFAULT_PARTITION__`), sedangkan SQLite menyimpannya sebagai teks `'None'` atau `'nan'` karena `astype(str)` di K-1.
Akibatnya baris "hasil sama?" di atas bisa `False` hanya karena **label**, bukan karena angkanya. Sel di bawah menyamakan label lalu membandingkan ulang.

In [8]:
def samakan_label(w):
    kosong = (w is None) or (isinstance(w, float) and pd.isna(w)) or w in ("nan", "None", "__HIVE_DEFAULT_PARTITION__")
    return "(kosong)" if kosong else w

norm_d = [(samakan_label(w), int(j), float(r)) for w, j, r in hasil_d.itertuples(index=False)]
norm_s = [(samakan_label(w), int(j), float(r)) for w, j, r in hasil_s]
hasil_sama_norm = norm_d == norm_s
print("label wilayah dari DuckDB :", [w for w in hasil_d["wilayah_jemput"]])
print("label wilayah dari SQLite :", [w for w, _, _ in hasil_s])
print("hasil sama setelah menyamakan label wilayah kosong?", hasil_sama_norm)
if not hasil_sama_norm:
    print("DuckDB:", norm_d)
    print("SQLite:", norm_s)

label wilayah dari DuckDB : ['Manhattan', 'Queens', 'Unknown', 'Brooklyn', 'Bronx', '__HIVE_DEFAULT_PARTITION__', 'Staten Island', 'EWR']
label wilayah dari SQLite : ['Manhattan', 'Queens', 'Unknown', 'Brooklyn', 'Bronx', 'None', 'Staten Island', 'EWR']
hasil sama setelah menyamakan label wilayah kosong? True


In [9]:
# Filter pada kolom partisi: DuckDB hanya membuka folder wilayah yang diminta (partition pruning, P3)
SQL2 = """SELECT COUNT(*), ROUND(AVG(total_bayar), 2)
          FROM perjalanan WHERE wilayah_jemput = 'Staten Island'"""
print(ukur("DuckDB - hanya wilayah Staten Island", lambda: con_d.sql(SQL2).fetchall()))

ukuran_pq = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(TRIPS_BERSIH) for f in fs) / 1024**2
ukuran_sqlite = os.path.getsize('baris.db') / 1024**2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom): {ukuran_sqlite:.1f} MB")

[DuckDB - hanya wilayah Staten Island] 0.0188 s
[(211, 74.07)]
Parquet (18 kolom): 66.5 MB | SQLite (5 kolom): 114.2 MB


## K-5. Graph: Pertanyaan tentang Keterhubungan

**Dugaan saya (isi SEBELUM menjalankan):**
- Untuk pertanyaan "zona yang bisa dicapai dalam 2 langkah", graph lebih cepat/lambat/sama dengan SQL self-join: ............
- Berapa zona yang terjangkau dalam <= 2 langkah dari zona terpopuler? ............

In [10]:
import networkx as nx

# Satu baris = satu arus: berapa perjalanan dari zona jemput ke zona tujuan tertentu
arus = ukur("hitung arus antar zona", lambda: con_d.sql("""
    SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
    FROM perjalanan GROUP BY 1, 2""").df())

# Zona = node (titik), arus = edge (garis berarah, bobotnya jumlah perjalanan)
G = nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))
print("node (zona):", G.number_of_nodes(), "| edge (arus):", G.number_of_edges())

# Lima arus terbesar. Urutan dibuat pasti dengan pengurut tambahan (jumlah, asal, tujuan)
top_graf = sorted(G.edges(data=True), key=lambda e: (-e[2]["jumlah"], e[0], e[1]))[:5]
print("graph:", [(a, b, d["jumlah"]) for a, b, d in top_graf])
top_sql = arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5)
print("SQL :", list(top_sql.itertuples(index=False, name=None)))

# Lima zona yang paling banyak menjadi tujuan
populer = sorted(G.in_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0]))[:5]
print("zona tujuan terpopuler:", populer)

[hitung arus antar zona] 0.1975 s
node (zona): 262 | edge (arus): 21583
graph: [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
SQL : [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
zona tujuan terpopuler: [(236, 144209), (237, 130369), (161, 113991), (230, 87706), (170, 87110)]


In [11]:
MIN = 50  # abaikan arus yang terlalu jarang (kurang dari 50 perjalanan)
zona_awal = populer[0][0]  # mulai dari zona tujuan terpopuler

# Pertanyaan: zona mana yang bisa dicapai dalam TEPAT 2 langkah dari zona_awal?
# (1) cara graph: telusuri tetangga dari tetangga
def dua_langkah_graph():
    hasil = set()
    for b, d1 in G[zona_awal].items():
        if d1["jumlah"] < MIN:
            continue
        for c, d2 in G[b].items():
            if d2["jumlah"] >= MIN and c != zona_awal:
                hasil.add(c)
    return hasil

# (2) cara SQL: harus menggabungkan tabel arus dengan dirinya sendiri (self-join)
con_d.register("arus", arus)
def dua_langkah_sql():
    return set(con_d.sql(f"""
        SELECT DISTINCT a2.tujuan
        FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
        WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN}
          AND a2.jumlah >= {MIN} AND a2.tujuan <> {zona_awal}""").df()["tujuan"])

hg = ukur("2 langkah - graph (NetworkX)", dua_langkah_graph)
hs = ukur("2 langkah - SQL self-join (DuckDB)", dua_langkah_sql)
print("zona awal:", zona_awal, "| terjangkau (graph):", len(hg), "| (SQL):", len(hs), "| sama?", hg == hs)

# (3) Jumlah langkah bebas: "dalam paling banyak k langkah" - cukup ganti angka k
H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
jangkauan_k = {}
for k in (1, 2, 3, 4):
    n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
    jangkauan_k[k] = n
    print(f"dalam <= {k} langkah: {n} zona")

[2 langkah - graph (NetworkX)] 0.0067 s
[2 langkah - SQL self-join (DuckDB)] 0.0137 s
zona awal: 236 | terjangkau (graph): 214 | (SQL): 214 | sama? True
dalam <= 1 langkah: 74 zona
dalam <= 2 langkah: 214 zona
dalam <= 3 langkah: 214 zona
dalam <= 4 langkah: 214 zona


## K-6. Indeks Spasial

> **Data pada bagian ini TIRUAN.** Data taksi 2023 hanya memuat nomor zona, tanpa koordinat. 200.000 titik jemput dibuat acak di sekitar New York.
> Angkanya menggambarkan cara kerja indeks, bukan pola perjalanan taksi yang sebenarnya.

**Dugaan saya (isi SEBELUM menjalankan):**
- STRtree dibanding loop Python: sekitar ......... kali lipat. STRtree dibanding NumPy: sekitar ......... kali lipat.

In [12]:
from shapely import STRtree, box, points

# CATATAN: data TLC 2023 hanya berisi ID zona, TANPA koordinat. Titik di bawah ini SINTETIS.
rng = np.random.default_rng(42)
N = 200_000
bujur = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70)   # longitude
lintang = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)    # latitude
titik = points(bujur, lintang)

# 1.000 wilayah pencarian berupa kotak kecil (sekitar 1 km x 1 km)
M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)
kotak = [box(x - 0.005, y - 0.005, x + 0.005, y + 0.005) for x, y in zip(cx, cy)]

# Cara 1: periksa titik satu per satu dengan loop Python (hanya 5 kotak, 1.000 akan sangat lama)
def cara_loop():
    return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

# Cara 2: bandingkan seluruh titik sekaligus dengan NumPy (tanpa indeks spasial), 1.000 kotak
def cara_numpy():
    hasil = []
    for x, y in zip(cx, cy):
        m = (bujur >= x - 0.005) & (bujur <= x + 0.005) & (lintang >= y - 0.005) & (lintang <= y + 0.005)
        hasil.append(int(m.sum()))
    return hasil

h_loop = ukur("cara 1 - loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 - NumPy, 1000 kotak", cara_numpy)

# Cara 3: indeks spasial STRtree (sejenis R-tree)
pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur("cara 3 - STRtree, 1000 kotak",
             lambda: [len(pohon.query(k, predicate="contains")) for k in kotak])
print("5 kotak pertama - loop:", h_loop, "| STRtree:", h_idx[:5])
print("hasil NumPy dan STRtree sama?", h_np == h_idx, "| loop dan STRtree sama (5 pertama)?", h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))

[cara 1 - loop Python, 5 kotak] 14.1278 s
[cara 2 - NumPy, 1000 kotak] 0.5384 s
[bangun indeks STRtree] 0.0723 s
[cara 3 - STRtree, 1000 kotak] 0.2049 s
5 kotak pertama - loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
hasil NumPy dan STRtree sama? True | loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 417545


## K-7. Delta Lake: Versi, Time Travel, dan Schema Enforcement

> `SchemaMismatchError` pada sel ke-3 di bawah **bukan kesalahan**; itu hasil yang diharapkan.

**Dugaan saya (isi SEBELUM menjalankan):**
- Setelah `update`, apakah data versi lama masih bisa dibaca? ............
- Apakah nomor versi berubah setelah `append` kolom asing yang ditolak? ............

In [13]:
from deltalake import DeltaTable, write_deltalake

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data_delta = df[KOLOM_DELTA].sample(n=min(200_000, len(df)), random_state=42).reset_index(drop=True)

PATH_DELTA = "/content/trips_delta"   # disk lokal Colab; ZIP-nya disalin ke Drive setelah K-8
if os.path.exists(PATH_DELTA):
    shutil.rmtree(PATH_DELTA)

# Versi 0: menulis data awal (100.000 baris)
write_deltalake(PATH_DELTA, data_delta.iloc[:100_000], mode="overwrite")
dt = DeltaTable(PATH_DELTA)
baris_v0 = len(dt.to_pandas())
print("versi:", dt.version(), "| baris:", baris_v0)

# Versi 1: menambah 50.000 baris
write_deltalake(PATH_DELTA, data_delta.iloc[100_000:150_000], mode="append")
dt = DeltaTable(PATH_DELTA)
baris_v1 = len(dt.to_pandas())
print("versi:", dt.version(), "| baris:", baris_v1)

# Versi 2: koreksi data. Anggap tarif Tunai di wilayah Bronx perlu ditambah 1.0
sekarang = data_delta.iloc[:150_000]
n_koreksi = int(((sekarang["wilayah_jemput"] == "Bronx") & (sekarang["metode_bayar"] == "Tunai")).sum())
print("baris yang akan dikoreksi:", n_koreksi)
dt.update(predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
          updates={"total_bayar": "total_bayar + 1.0"})
print("versi:", DeltaTable(PATH_DELTA).version())

versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 20
versi: 2


In [14]:
# TIME TRAVEL: baca tabel sebagaimana keadaannya pada versi tertentu
v0 = DeltaTable(PATH_DELTA, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA).to_pandas()
print("baris v0:", len(v0), "| v1:", len(v1), "| terbaru:", len(terbaru))
selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print("selisih total_bayar (terbaru - v1):", round(selisih, 2), "| jumlah baris dikoreksi:", n_koreksi)

# Riwayat perubahan tabel
for h in DeltaTable(PATH_DELTA).history():
    print(h.get("version"), h.get("operation"))

# Log transaksi: satu file JSON per versi
print(sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

baris v0: 100000 | v1: 150000 | terbaru: 150000
selisih total_bayar (terbaru - v1): 20.0 | jumlah baris dikoreksi: 20
2 UPDATE
1 WRITE
0 WRITE
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']


In [15]:
# PENEGAKAN SKEMA: menambah data dengan kolom asing harus DITOLAK, tabel tidak berubah
tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1
try:
    write_deltalake(PATH_DELTA, tambahan, mode="append")
    nama_error = None
    print("append kolom asing: BERHASIL (tidak diharapkan)")
except Exception as e:
    nama_error = type(e).__name__
    print("append kolom asing DITOLAK:", nama_error)
versi_setelah_gagal = DeltaTable(PATH_DELTA).version()
print("versi setelah percobaan gagal:", versi_setelah_gagal)

append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2


## K-8. Small File Problem pada Tabel Delta

**Dugaan saya (isi SEBELUM menjalankan):**
- Setelah 20 append kecil, jumlah file aktif menjadi sekitar ......... ; setelah compaction menjadi ......... .
- Setelah compaction, jumlah file di disk lebih sedikit / sama / lebih banyak dari file aktif: ............

In [16]:
dt = DeltaTable(PATH_DELTA)
n_aktif_awal = len(dt.file_uris())
print("file data aktif sebelum:", n_aktif_awal, "| versi:", dt.version())

# 20 kali menambah data kecil (500 baris) - seperti data yang masuk sedikit demi sedikit
for i in range(20):
    potongan = data_delta.iloc[150_000 + i * 500: 150_000 + (i + 1) * 500]
    write_deltalake(PATH_DELTA, potongan, mode="append")
dt = DeltaTable(PATH_DELTA)
n_aktif_append = len(dt.file_uris())
print("file data aktif setelah 20 append kecil:", n_aktif_append, "| versi:", dt.version(),
      "| baris:", len(dt.to_pandas()))
t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sebelum = time.perf_counter() - t0

# COMPACTION: gabungkan file kecil menjadi file yang lebih besar (dicatat sebagai versi baru)
hasil_compact = dt.optimize.compact()
print("hasil compact:", {k: hasil_compact[k] for k in ("numFilesAdded", "numFilesRemoved")})
dt = DeltaTable(PATH_DELTA)
t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sesudah = time.perf_counter() - t0
n_aktif_compact = len(dt.file_uris())
baris_akhir_k8 = len(dt.to_pandas())
print("file data aktif setelah compact:", n_aktif_compact, "| versi:", dt.version(), "| baris:", baris_akhir_k8)
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")
catatan.append({"langkah": "K-8 baca tabel Delta sebelum compact", "detik": round(t_sebelum, 4)})
catatan.append({"langkah": "K-8 baca tabel Delta sesudah compact", "detik": round(t_sesudah, 4)})

# File lama TIDAK langsung terhapus (masih dibutuhkan untuk time travel). VACUUM yang menghapusnya.
n_parquet_disk = len(glob.glob(f"{PATH_DELTA}/*.parquet"))
print("file parquet di disk:", n_parquet_disk)
kandidat = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True)  # dry_run = hanya melaporkan
n_vacuum_dry = len(kandidat)
print("file yang akan dihapus VACUUM (dry run):", n_vacuum_dry)

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1 | versi: 23 | baris: 160000
waktu baca sebelum compact: 0.0770 s | sesudah: 0.0529 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


**PERINGATAN:** `retention_hours=0` bersama `dry_run=True` hanya untuk melihat cakupan VACUUM tanpa menghapus apa pun.
Jangan menjalankan VACUUM retention 0 jam dengan `dry_run=False` pada tabel sungguhan.

### Simpan `trips_delta/` ke Drive (dibuat SEBELUM Latihan 6, karena `restore` menambah versi)

In [17]:
ZIP_DELTA = shutil.make_archive(f"{DIR_SIMPAN}/trips_delta", "zip", PATH_DELTA)
print("tersimpan:", ZIP_DELTA, f"({os.path.getsize(ZIP_DELTA)/1024**2:.2f} MB)")
print("versi tabel yang diarsipkan:", DeltaTable(PATH_DELTA).version())
print("isi _delta_log:", len(os.listdir(f"{PATH_DELTA}/_delta_log")), "berkas")

tersimpan: /content/drive/MyDrive/BigData/Praktikum4/trips_delta.zip (5.72 MB)
versi tabel yang diarsipkan: 23
isi _delta_log: 24 berkas


## K-9. Merangkum dan Memilih

In [18]:
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print(rekap.to_string(index=False))

                                            langkah   detik
                                  baca trips_bersih  4.8379
                             relasional: muat tabel  4.0217
                             key-value: bangun dict  7.1633
             cari 200 perjalanan - SQL tanpa indeks  4.2453
            cari 200 perjalanan - SQL dengan indeks  0.0038
                    cari 200 perjalanan - key-value  0.0012
                  hitung per wilayah - SQL GROUP BY  0.1915
  hitung per wilayah - key-value (buka semua nilai)  1.4724
DuckDB - hitung per wilayah (langsung dari Parquet)  0.1158
               SQLite - muat seluruh baris ke tabel  9.2370
                        SQLite - hitung per wilayah  1.7315
               DuckDB - hanya wilayah Staten Island  0.0188
                             hitung arus antar zona  0.1975
                       2 langkah - graph (NetworkX)  0.0067
                 2 langkah - SQL self-join (DuckDB)  0.0137
                      cara 1 - loop Pyth

**Tabel rekap K-9 (terisi otomatis dari angka sendiri).** Rasio dihitung dari waktu tanpa pembulatan.

In [19]:
W = waktu_mentah
def rasio(a, b):
    return a / max(b, 1e-9)

t_noidx = W["cari 200 perjalanan - SQL tanpa indeks"]
t_idx = W["cari 200 perjalanan - SQL dengan indeks"]
t_kv = W["cari 200 perjalanan - key-value"]
t_h_sql = W["hitung per wilayah - SQL GROUP BY"]
t_h_kv = W["hitung per wilayah - key-value (buka semua nilai)"]
t_duck = W["DuckDB - hitung per wilayah (langsung dari Parquet)"]
t_sqlite = W["SQLite - hitung per wilayah"]
t_g2 = W["2 langkah - graph (NetworkX)"]
t_s2 = W["2 langkah - SQL self-join (DuckDB)"]
t_loop5 = W["cara 1 - loop Python, 5 kotak"]
t_np = W["cara 2 - NumPy, 1000 kotak"]
t_bangun = W["bangun indeks STRtree"]
t_tree = W["cara 3 - STRtree, 1000 kotak"]
t_loop1000_est = t_loop5 / 5 * 1000

tabel_rekap = pd.DataFrame([
    {"Kebutuhan": "Ambil satu data lewat id",
     "Unggul": "Key-value; tabel dengan indeks", "Lemah": "Tabel tanpa indeks",
     "Angka saya": (f"tanpa indeks {t_noidx:.4f} s | berindeks {t_idx:.4f} s | key-value {t_kv:.4f} s "
                    f"(tanpa/dengan indeks = {rasio(t_noidx, t_idx):,.0f}x)")},
    {"Kebutuhan": "Menghitung/merangkum banyak baris",
     "Unggul": "Per kolom (DuckDB pada Parquet)", "Lemah": "Key-value (buka semua nilai)",
     "Angka saya": (f"DuckDB {t_duck:.4f} s | SQLite {t_sqlite:.4f} s ({rasio(t_sqlite, t_duck):.1f}x) | "
                    f"SQL 300rb baris {t_h_sql:.4f} s vs key-value {t_h_kv:.4f} s ({rasio(t_h_kv, t_h_sql):.1f}x)")},
    {"Kebutuhan": "Data dengan field opsional/berubah-ubah",
     "Unggul": "Document", "Lemah": "Tabel (kolom NULL, migrasi)",
     "Angka saya": f"(kualitatif) {n_dok_tanpa_tip:,} dari {n_dok_total - 1:,} dokumen tanpa field tip; 1 dokumen membawa field cuaca"},
    {"Kebutuhan": "Keterhubungan dengan jumlah langkah bebas",
     "Unggul": "Graph", "Lemah": "SQL (satu JOIN per langkah)",
     "Angka saya": (f"2 langkah: graph {t_g2:.4f} s | SQL {t_s2:.4f} s | jangkauan k=1..4: "
                    f"{[jangkauan_k[k] for k in (1, 2, 3, 4)]}")},
    {"Kebutuhan": "Mencari lokasi dalam wilayah",
     "Unggul": "Spasial dengan indeks", "Lemah": "Memeriksa semua titik",
     "Angka saya": (f"loop ~{t_loop1000_est:.0f} s (perkiraan 1.000 kotak) | NumPy {t_np:.3f} s | "
                    f"STRtree {t_bangun + t_tree:.3f} s (indeks {t_bangun:.3f} + cari {t_tree:.3f})")},
    {"Kebutuhan": "Perubahan yang harus aman dan bisa dibatalkan",
     "Unggul": "Tabel Delta", "Lemah": "Folder Parquet biasa",
     "Angka saya": (f"(kualitatif) update mengubah {n_koreksi:,} baris, v0/v1 tetap terbaca; "
                    f"append kolom asing ditolak ({nama_error}), versi tetap {versi_setelah_gagal}")},
])
pd.set_option("display.max_colwidth", None)
display(tabel_rekap)
tabel_rekap.to_csv(f"{DIR_SIMPAN}/tabel_rekap_k9.csv", index=False)

,Kebutuhan,Unggul,Lemah,Angka saya
0,Ambil satu data lewat id,Key-value; tabel dengan indeks,Tabel tanpa indeks,"tanpa indeks 4.2453 s | berindeks 0.0038 s | key-value 0.0012 s (tanpa/dengan indeks = 1,110x)"
1,Menghitung/merangkum banyak baris,Per kolom (DuckDB pada Parquet),Key-value (buka semua nilai),DuckDB 0.1158 s | SQLite 1.7315 s (14.9x) | SQL 300rb baris 0.1915 s vs key-value 1.4724 s (7.7x)
2,Data dengan field opsional/berubah-ubah,Document,"Tabel (kolom NULL, migrasi)","(kualitatif) 4,094 dari 20,000 dokumen tanpa field tip; 1 dokumen membawa field cuaca"
3,Keterhubungan dengan jumlah langkah bebas,Graph,SQL (satu JOIN per langkah),"2 langkah: graph 0.0067 s | SQL 0.0137 s | jangkauan k=1..4: [74, 214, 214, 214]"
4,Mencari lokasi dalam wilayah,Spasial dengan indeks,Memeriksa semua titik,loop ~2826 s (perkiraan 1.000 kotak) | NumPy 0.538 s | STRtree 0.277 s (indeks 0.072 + cari 0.205)
5,Perubahan yang harus aman dan bisa dibatalkan,Tabel Delta,Folder Parquet biasa,"(kualitatif) update mengubah 20 baris, v0/v1 tetap terbaca; append kolom asing ditolak (SchemaMismatchError), versi tetap 2"


**Angka untuk Analisis Hasil (Bagian O).** Sel ini hanya menghitung; interpretasi ditulis di laporan.

In [20]:
print("O.1  tanpa indeks / berindeks      :", f"{rasio(t_noidx, t_idx):,.1f}x")
print("     berindeks / key-value         :", f"{rasio(t_idx, t_kv):.2f}x  (selisih {abs(t_idx - t_kv) * 1000:.3f} ms untuk 200 pencarian)")
print("O.2  key-value / SQL GROUP BY      :", f"{rasio(t_h_kv, t_h_sql):.1f}x")
print("O.4  SQLite / DuckDB (hitung)      :", f"{rasio(t_sqlite, t_duck):.1f}x")
print("     SQLite muat + hitung / DuckDB :", f"{rasio(W['SQLite - muat seluruh baris ke tabel'] + t_sqlite, t_duck):.1f}x")
print("     ukuran Parquet vs SQLite (MB) :", f"{ukuran_pq:.1f} vs {ukuran_sqlite:.1f}")
print("O.5  graph vs SQL (2 langkah)      :", f"{t_g2:.4f} s vs {t_s2:.4f} s")
print("O.6  perkiraan loop 1.000 kotak    :", f"{t_loop1000_est:.0f} s")
print("     loop / STRtree (indeks + cari):", f"{rasio(t_loop1000_est, t_bangun + t_tree):,.0f}x")
print("     NumPy / STRtree (indeks + cari):", f"{rasio(t_np, t_bangun + t_tree):.2f}x")
print("     NumPy / STRtree (cari saja)   :", f"{rasio(t_np, t_tree):.2f}x")
print("O.7  baris v0 / v1 / terbaru       :", baris_v0, "/", baris_v1, "/", len(terbaru))
print("     file aktif: awal / setelah 20 append / setelah compact:", n_aktif_awal, "/", n_aktif_append, "/", n_aktif_compact)
print("     file parquet di disk:", n_parquet_disk, "| kandidat VACUUM (dry run):", n_vacuum_dry)

O.1  tanpa indeks / berindeks      : 1,109.5x
     berindeks / key-value         : 3.21x  (selisih 2.635 ms untuk 200 pencarian)
O.2  key-value / SQL GROUP BY      : 7.7x
O.4  SQLite / DuckDB (hitung)      : 14.9x
     SQLite muat + hitung / DuckDB : 94.7x
     ukuran Parquet vs SQLite (MB) : 66.5 vs 114.2
O.5  graph vs SQL (2 langkah)      : 0.0067 s vs 0.0137 s
O.6  perkiraan loop 1.000 kotak    : 2826 s
     loop / STRtree (indeks + cari): 10,195x
     NumPy / STRtree (indeks + cari): 1.94x
     NumPy / STRtree (cari saja)   : 2.63x
O.7  baris v0 / v1 / terbaru       : 100000 / 150000 / 150000
     file aktif: awal / setelah 20 append / setelah compact: 1 / 21 / 1
     file parquet di disk: 24 | kandidat VACUUM (dry run): 23


## Latihan 1 — Pencarian 2.000 id acak

In [21]:
random.seed(42)
id_uji2 = random.sample(range(1, len(data) + 1), 2000)

def cari_sql_n():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji2]

def cari_kv_n():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji2]

ukur_median("L1: 2000 pencarian - SQL berindeks", cari_sql_n)
ukur_median("L1: 2000 pencarian - key-value", cari_kv_n)
mikro_sql = waktu_mentah["L1: 2000 pencarian - SQL berindeks"] / 2000 * 1e6
mikro_kv = waktu_mentah["L1: 2000 pencarian - key-value"] / 2000 * 1e6
print(f"rata-rata per pencarian | SQL berindeks: {mikro_sql:.2f} mikrodetik | key-value: {mikro_kv:.2f} mikrodetik")

[L1: 2000 pencarian - SQL berindeks] median 5x: 0.035321 s
[L1: 2000 pencarian - key-value] median 5x: 0.011288 s
rata-rata per pencarian | SQL berindeks: 17.66 mikrodetik | key-value: 5.64 mikrodetik


## Latihan 2 — Dokumen vs SQL untuk zona tujuan tertentu dan "Tunai"
Dokumen K-3 hanya berisi 20.000 perjalanan pertama contoh (id 1–20.000), sedangkan tabel SQL K-2 berisi 300.000. Karena itu SQL dihitung dua kali: seluruh tabel, dan dibatasi `id_perjalanan <= 20000`.

In [22]:
zona_uji = int(sample["zona_tujuan"].value_counts().index[0])   # zona tujuan terbanyak pada contoh
n_doc = len(tabel.search((Q.tujuan.zona == zona_uji) & (Q.biaya.metode_bayar == "Tunai")))
n_sql_semua = con.execute("SELECT COUNT(*) FROM perjalanan WHERE zona_tujuan = ? AND metode_bayar = 'Tunai'",
                          (zona_uji,)).fetchone()[0]
n_sql_20k = con.execute("SELECT COUNT(*) FROM perjalanan WHERE zona_tujuan = ? AND metode_bayar = 'Tunai' "
                        "AND id_perjalanan <= 20000", (zona_uji,)).fetchone()[0]
print("zona tujuan uji:", zona_uji)
print("TinyDB (20.001 dokumen)          :", n_doc)
print("SQL seluruh tabel (300.000 baris):", n_sql_semua)
print("SQL id <= 20000 (himpunan sama)  :", n_sql_20k, "| sama dengan dokumen?", n_doc == n_sql_20k)

zona tujuan uji: 236
TinyDB (20.001 dokumen)          : 155
SQL seluruh tabel (300.000 baris): 2074
SQL id <= 20000 (himpunan sama)  : 155 | sama dengan dokumen? True


## Latihan 3 — SUM satu kolom vs beberapa kolom (column pruning)
Memakai `read_parquet` langsung (nama kolom asli) agar ada cukup banyak kolom angka untuk dibandingkan.

In [23]:
SATU = "SELECT SUM(total_amount) FROM read_parquet('{g}', hive_partitioning=true)".format(g=GLOB)
KOL_ANGKA = ["total_amount", "trip_distance", "fare_amount", "tip_amount", "durasi_menit", "suhu_c", "hujan_mm"]
BANYAK = "SELECT " + ", ".join(f"SUM({c})" for c in KOL_ANGKA) + \
         " FROM read_parquet('{g}', hive_partitioning=true)".format(g=GLOB)
con_d.sql(SATU).fetchall()   # pemanasan (cache berkas)
ukur_median("L3: SUM 1 kolom (DuckDB)", lambda: con_d.sql(SATU).fetchall())
ukur_median(f"L3: SUM {len(KOL_ANGKA)} kolom (DuckDB)", lambda: con_d.sql(BANYAK).fetchall())
t1 = waktu_mentah["L3: SUM 1 kolom (DuckDB)"]
t7 = waktu_mentah[f"L3: SUM {len(KOL_ANGKA)} kolom (DuckDB)"]
print(f"1 kolom: {t1:.4f} s | {len(KOL_ANGKA)} kolom: {t7:.4f} s | rasio: {rasio(t7, t1):.2f}x")

[L3: SUM 1 kolom (DuckDB)] median 5x: 0.069133 s
[L3: SUM 7 kolom (DuckDB)] median 5x: 0.230922 s
1 kolom: 0.0691 s | 7 kolom: 0.2309 s | rasio: 3.34x


## Latihan 4 — Lima zona dengan tujuan berbeda terbanyak (out-degree vs COUNT DISTINCT)

In [24]:
top_out = sorted(G.out_degree(), key=lambda kv: (-kv[1], kv[0]))[:5]
top_out_sql = con_d.sql("""
    SELECT asal, COUNT(DISTINCT tujuan) AS n_tujuan FROM arus
    GROUP BY asal ORDER BY n_tujuan DESC, asal LIMIT 5""").fetchall()
print("graph (out-degree):", top_out)
print("SQL (COUNT DISTINCT):", [(int(a), int(b)) for a, b in top_out_sql])
print("sama?", [(a, b) for a, b in top_out] == [(int(a), int(b)) for a, b in top_out_sql])

graph (out-degree): [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
SQL (COUNT DISTINCT): [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
sama? True


## Latihan 5 — Ukuran kotak 0,02 dan 0,001 derajat
Sisi kotak pada K-6 adalah 0,01 derajat. Di sini diulang dengan sisi 0,02 dan 0,001. Titik tiruan sama (lihat catatan K-6).

In [25]:
def bandingkan_sisi(sisi, label):
    setengah = sisi / 2
    kotak_s = [box(x - setengah, y - setengah, x + setengah, y + setengah) for x, y in zip(cx, cy)]
    def np_s():
        return [int(((bujur >= x - setengah) & (bujur <= x + setengah) &
                     (lintang >= y - setengah) & (lintang <= y + setengah)).sum()) for x, y in zip(cx, cy)]
    def tree_s():
        return [len(pohon.query(k, predicate="contains")) for k in kotak_s]
    h_n = ukur(f"L5 NumPy, sisi {label}", np_s)
    h_t = ukur(f"L5 STRtree cari, sisi {label}", tree_s)
    t_n = waktu_mentah[f"L5 NumPy, sisi {label}"]
    t_t = waktu_mentah[f"L5 STRtree cari, sisi {label}"]
    print(f"   sisi {label}: hasil sama? {h_n == h_t} | total titik {sum(h_t):,} | "
          f"NumPy/STRtree (cari saja) {rasio(t_n, t_t):.2f}x | NumPy/STRtree (+indeks {t_bangun:.3f}s) {rasio(t_n, t_t + t_bangun):.2f}x")

bandingkan_sisi(0.02, "0.02")
print(f"   (K-6, sisi 0.01: NumPy/STRtree cari saja {rasio(t_np, t_tree):.2f}x | +indeks {rasio(t_np, t_tree + t_bangun):.2f}x)")
bandingkan_sisi(0.001, "0.001")

[L5 NumPy, sisi 0.02] 0.5386 s
[L5 STRtree cari, sisi 0.02] 0.7486 s
   sisi 0.02: hasil sama? True | total titik 1,666,631 | NumPy/STRtree (cari saja) 0.72x | NumPy/STRtree (+indeks 0.072s) 0.66x
   (K-6, sisi 0.01: NumPy/STRtree cari saja 2.63x | +indeks 1.94x)
[L5 NumPy, sisi 0.001] 0.7098 s
[L5 STRtree cari, sisi 0.001] 0.0194 s
   sisi 0.001: hasil sama? True | total titik 4,149 | NumPy/STRtree (cari saja) 36.65x | NumPy/STRtree (+indeks 0.072s) 7.74x


## Latihan 6 — `restore` vs time travel
> Sel ini mengubah tabel `/content/trips_delta` (versi baru). ZIP di Drive sudah dibuat sebelumnya, jadi aman.

In [26]:
dt = DeltaTable(PATH_DELTA)
print("versi sebelum restore:", dt.version(), "| baris:", len(dt.to_pandas()))
dt.restore(1)   # kembalikan tabel ke keadaan versi 1 (dicatat sebagai versi baru)
dt = DeltaTable(PATH_DELTA)
print("versi setelah restore:", dt.version(), "| baris:", len(dt.to_pandas()))
print([h.get("operation") for h in dt.history()][:3])

versi sebelum restore: 23 | baris: 160000
versi setelah restore: 24 | baris: 150000
['RESTORE', 'OPTIMIZE', 'WRITE']


## Tugas R.2 — Data lain, cara lain: **zona**
Satu data (zona taksi) disimpan dengan tiga cara: tabel (SQLite), document (TinyDB), dan graph (NetworkX).
Untuk tiap cara dicoba satu pertanyaan yang mudah dan satu yang canggung.

In [27]:
# Bahan: referensi zona -> wilayah (dari data) dan tabel arus K-5
zona_ref = (df[["zona_jemput", "wilayah_jemput"]].drop_duplicates("zona_jemput")
            .rename(columns={"zona_jemput": "id_zona", "wilayah_jemput": "wilayah"})
            .sort_values("id_zona").reset_index(drop=True))
print("zona unik:", len(zona_ref), "| arus:", len(arus))

# --- Cara 1: TABEL (SQLite) ---
con_z = sqlite3.connect(":memory:")
zona_ref.to_sql("zona", con_z, index=False)
arus.to_sql("arus", con_z, index=False)
con_z.execute("CREATE INDEX idx_arus_asal ON arus(asal)")
z0 = int(zona_awal)

# mudah: wilayah sebuah zona
mudah_tabel = con_z.execute("SELECT wilayah FROM zona WHERE id_zona = ?", (z0,)).fetchone()
print("[tabel/mudah] wilayah zona", z0, "=", mudah_tabel)

# canggung: zona yang terjangkau dalam <= 3 langkah (dua self-join tambahan)
def tiga_langkah_sql():
    return {r[0] for r in con_z.execute(f"""
        SELECT a1.tujuan FROM arus a1 WHERE a1.asal = {z0} AND a1.jumlah >= {MIN}
        UNION SELECT a2.tujuan FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
              WHERE a1.asal = {z0} AND a1.jumlah >= {MIN} AND a2.jumlah >= {MIN}
        UNION SELECT a3.tujuan FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
              JOIN arus a3 ON a2.tujuan = a3.asal
              WHERE a1.asal = {z0} AND a1.jumlah >= {MIN} AND a2.jumlah >= {MIN} AND a3.jumlah >= {MIN}""")} - {z0}
hasil_t3 = ukur("R2 tabel: <=3 langkah (3 SELECT + 2 self-join)", tiga_langkah_sql)

# --- Cara 2: DOCUMENT (TinyDB): satu dokumen per zona, 5 tujuan teratas disematkan (embedding) ---
db_z = TinyDB(storage=MemoryStorage)
dok_zona = db_z.table("zona")
top5 = (arus.sort_values(["asal", "jumlah", "tujuan"], ascending=[True, False, True])
        .groupby("asal").head(5))
wilayah_dari = dict(zip(zona_ref["id_zona"], zona_ref["wilayah"]))
dok_zona.insert_multiple([
    {"id_zona": int(a), "wilayah": wilayah_dari.get(a),
     "keluar_total": int(arus.loc[arus["asal"] == a, "jumlah"].sum()),
     "top_tujuan": [{"zona": int(t), "jumlah": int(j)} for t, j in zip(g["tujuan"], g["jumlah"])]}
    for a, g in top5.groupby("asal")])
print("dokumen zona:", len(dok_zona))

# mudah: 5 tujuan teratas dari sebuah zona (satu dokumen, satu bacaan)
mudah_doc = dok_zona.search(Q.id_zona == z0)[0]["top_tujuan"]
print("[document/mudah] 5 tujuan teratas dari", z0, ":", [(d["zona"], d["jumlah"]) for d in mudah_doc])

# canggung: dari zona mana saja sebuah zona termasuk 5 tujuan teratas (arah terbalik -> pindai semua dokumen)
def arah_terbalik():
    return [d["id_zona"] for d in dok_zona.all() if any(t["zona"] == z0 for t in d["top_tujuan"])]
hasil_balik = ukur("R2 document: pindai semua dokumen (arah terbalik)", arah_terbalik)
n_asal_sebenarnya = int(arus.loc[arus["tujuan"] == z0, "asal"].nunique())
print("[document/canggung] zona yang menempatkan", z0, "di top-5 tujuannya:", len(hasil_balik), "zona |",
      "zona asal yang SEBENARNYA mengirim perjalanan ke sana (tabel arus):", n_asal_sebenarnya,
      "-> jawaban dokumen tidak lengkap karena hanya top-5 yang disematkan")

# --- Cara 3: GRAPH (NetworkX, graph G dari K-5) ---
mudah_graph = len(nx.single_source_shortest_path_length(H, z0, cutoff=3)) - 1
print("[graph/mudah] zona terjangkau <= 3 langkah dari", z0, ":", mudah_graph, "| tabel/SQL:", len(hasil_t3),
      "| sama?", mudah_graph == len(hasil_t3))
atribut_edge = sorted({k for _, _, d in G.edges(data=True) for k in d})
print("[graph/canggung] atribut yang tersimpan pada edge:", atribut_edge,
      "-> rata-rata total_bayar per wilayah tidak bisa dijawab tanpa menambah atribut atau memuat ulang data perjalanan")

zona unik: 254 | arus: 21583
[tabel/mudah] wilayah zona 236 = ('Manhattan',)
[R2 tabel: <=3 langkah (3 SELECT + 2 self-join)] 0.5625 s
dokumen zona: 254
[document/mudah] 5 tujuan teratas dari 236 : [(237, 18810), (236, 14110), (161, 7497), (239, 6877), (141, 6876)]
[R2 document: pindai semua dokumen (arah terbalik)] 0.0009 s
[document/canggung] zona yang menempatkan 236 di top-5 tujuannya: 23 zona | zona asal yang SEBENARNYA mengirim perjalanan ke sana (tabel arus): 133 -> jawaban dokumen tidak lengkap karena hanya top-5 yang disematkan
[graph/mudah] zona terjangkau <= 3 langkah dari 236 : 214 | tabel/SQL: 214 | sama? True
[graph/canggung] atribut yang tersimpan pada edge: ['jumlah'] -> rata-rata total_bayar per wilayah tidak bisa dijawab tanpa menambah atribut atau memuat ulang data perjalanan


## Tugas R.3 — Delta Lake dua hari
Hari 1: menulis sebagian data. Hari 2: menambah data dan mengoreksi beberapa baris.
Dibuktikan bahwa tabel "pada akhir hari 1" bisa dibaca lagi lewat time travel, lalu isi `_delta_log/` dilampirkan.
Memakai tabel terpisah (`/content/trips_delta_2hari`) supaya tidak bercampur dengan K-7/K-8.

In [28]:
PATH_2HARI = "/content/trips_delta_2hari"
shutil.rmtree(PATH_2HARI, ignore_errors=True)
data_2h = df[KOLOM_DELTA].sample(n=min(120_000, len(df)), random_state=7).reset_index(drop=True)

# ===== HARI 1: dua penulisan =====
write_deltalake(PATH_2HARI, data_2h.iloc[:60_000], mode="overwrite")            # versi 0
write_deltalake(PATH_2HARI, data_2h.iloc[60_000:80_000], mode="append")         # versi 1
versi_akhir_hari1 = DeltaTable(PATH_2HARI).version()
tabel_hari1 = DeltaTable(PATH_2HARI).to_pandas()
baris_hari1, total_hari1 = len(tabel_hari1), float(tabel_hari1["total_bayar"].sum())
print(f"AKHIR HARI 1 | versi {versi_akhir_hari1} | baris {baris_hari1:,} | jumlah total_bayar {total_hari1:,.2f}")

# ===== HARI 2: tambah data, lalu koreksi =====
write_deltalake(PATH_2HARI, data_2h.iloc[80_000:120_000], mode="append")        # versi 2
sekarang2 = data_2h.iloc[:120_000]
n_koreksi2 = int(((sekarang2["wilayah_jemput"] == "Queens") & (sekarang2["metode_bayar"] == "Tunai")).sum())
dt2 = DeltaTable(PATH_2HARI)
dt2.update(predicate="wilayah_jemput = 'Queens' AND metode_bayar = 'Tunai'",
           updates={"total_bayar": "total_bayar + 2.0"})                         # versi 3
versi_akhir_hari2 = DeltaTable(PATH_2HARI).version()
tabel_hari2 = DeltaTable(PATH_2HARI).to_pandas()
print(f"AKHIR HARI 2 | versi {versi_akhir_hari2} | baris {len(tabel_hari2):,} | "
      f"jumlah total_bayar {tabel_hari2['total_bayar'].sum():,.2f} | baris dikoreksi {n_koreksi2:,}")

# ===== BUKTI: baca tabel seperti akhir hari 1 =====
balik = DeltaTable(PATH_2HARI, version=versi_akhir_hari1).to_pandas()
print(f"time travel ke versi {versi_akhir_hari1}: baris {len(balik):,} | jumlah total_bayar {balik['total_bayar'].sum():,.2f}")
bukti_hari1 = (len(balik) == baris_hari1) and abs(float(balik["total_bayar"].sum()) - total_hari1) < 0.01
print("sama persis dengan keadaan akhir hari 1?", bukti_hari1)

print("\nRiwayat:")
for h in DeltaTable(PATH_2HARI).history():
    print("  versi", h.get("version"), "-", h.get("operation"))

AKHIR HARI 1 | versi 1 | baris 80,000 | jumlah total_bayar 2,185,789.40
AKHIR HARI 2 | versi 3 | baris 120,000 | jumlah total_bayar 3,279,036.55 | baris dikoreksi 2,593
time travel ke versi 1: baris 80,000 | jumlah total_bayar 2,185,789.40
sama persis dengan keadaan akhir hari 1? True

Riwayat:
  versi 3 - UPDATE
  versi 2 - WRITE
  versi 1 - WRITE
  versi 0 - WRITE


In [29]:
# ===== LAMPIRAN: isi _delta_log/ =====
DIR_LOG = f"{PATH_2HARI}/_delta_log"
berkas_log = sorted(f for f in os.listdir(DIR_LOG) if f.endswith(".json"))
print("berkas log:", berkas_log, "\n")

def ringkas_log(nama):
    ringkasan = []
    with open(os.path.join(DIR_LOG, nama)) as f:
        for baris in f:
            aksi = json.loads(baris)
            jenis = next(iter(aksi))
            isi = aksi[jenis]
            if jenis == "commitInfo":
                ringkasan.append(f"commitInfo  operation={isi.get('operation')}")
            elif jenis == "add":
                stat = json.loads(isi["stats"]) if isi.get("stats") else {}
                ringkasan.append(f"add         {isi['path']}  size={isi['size']}  numRecords={stat.get('numRecords')}")
            elif jenis == "remove":
                ringkasan.append(f"remove      {isi['path']}")
            elif jenis == "metaData":
                ringkasan.append(f"metaData    kolom={[c['name'] for c in json.loads(isi['schemaString'])['fields']]}")
            else:
                ringkasan.append(jenis)
    return ringkasan

teks_lampiran = []
for nama in berkas_log:
    teks_lampiran.append(f"=== {nama} ===")
    teks_lampiran += ["  " + r for r in ringkas_log(nama)]
print("\n".join(teks_lampiran))

# versi lengkap (JSON mentah) disimpan ke Drive sebagai lampiran
with open(f"{DIR_SIMPAN}/delta_log_2hari.txt", "w") as out:
    out.write("RINGKASAN\n" + "\n".join(teks_lampiran) + "\n\nISI MENTAH\n")
    for nama in berkas_log:
        out.write(f"\n=== {nama} ===\n")
        out.write(open(os.path.join(DIR_LOG, nama)).read())
zip_2hari = shutil.make_archive(f"{DIR_SIMPAN}/trips_delta_2hari", "zip", PATH_2HARI)
print("\ntersimpan: delta_log_2hari.txt dan", os.path.basename(zip_2hari))

berkas log: ['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json', '00000000000000000003.json'] 

=== 00000000000000000000.json ===
  commitInfo  operation=WRITE
  protocol
  metaData    kolom=['waktu_jemput', 'zona_jemput', 'zona_tujuan', 'wilayah_jemput', 'jarak_mil', 'total_bayar', 'metode_bayar']
  add         part-00000-2a8ab66e-d7d9-4b16-978a-e69932c8ce18-c000.snappy.parquet  size=908058  numRecords=60000
=== 00000000000000000001.json ===
  commitInfo  operation=WRITE
  add         part-00000-c6803a47-9309-4174-9bd7-5e828fb0d7cd-c000.snappy.parquet  size=311947  numRecords=20000
=== 00000000000000000002.json ===
  commitInfo  operation=WRITE
  add         part-00000-124cb90e-48a3-4a55-a3fe-09c2f4f3cdad-c000.snappy.parquet  size=610135  numRecords=40000
=== 00000000000000000003.json ===
  commitInfo  operation=UPDATE
  add         part-00000-5993afa7-6803-4d5b-9aec-ca88d2b7db41-c000.snappy.parquet  size=1795678  numRecords=120000
  remove      part

## Sel penutup
Menyimpan ulang `benchmark_nosql.csv` (kini memuat juga Latihan dan R.2) dan memeriksa berkas keluaran.

In [30]:
pd.DataFrame(catatan).to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print("baris benchmark_nosql.csv:", len(catatan))
print(f"baris trips_bersih/ yang dibaca: {len(df):,}")
for f in sorted(os.listdir(DIR_SIMPAN)):
    p = os.path.join(DIR_SIMPAN, f)
    if os.path.isfile(p):
        print(f"{f:35s} {os.path.getsize(p)/1024**2:8.2f} MB")

baris benchmark_nosql.csv: 31
baris trips_bersih/ yang dibaca: 2,986,910
benchmark_nosql.csv                     0.00 MB
delta_log_2hari.txt                     0.01 MB
tabel_rekap_k9.csv                      0.00 MB
trips_delta.zip                         5.72 MB
trips_delta_2hari.zip                   3.01 MB
